In [4]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from medmnist import INFO, PathMNIST

# ---------------------------------------------------------
# Configuration (Direct Paths - No CLI Args)
# ---------------------------------------------------------
NPZ_PATH = "./v3/pathmnist.npz"
STD_CKPT_PATH = "./v3/best_medlit_mod_std.pth"
ADV_CKPT_PATH = "./v3/best_medlit_mod_adv.pth"
OUTPUT_IMAGE = "pathmnist_fgsm_both_models_demo.png"

EPSILON = 0.05
NUM_SAMPLES = 4
BATCH_SIZE = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# ---------------------------------------------------------
# 1. Exact MedLiT-Seed + MoD Model Architecture
# ---------------------------------------------------------
class SwiGLUExpert(nn.Module):
    def __init__(self, in_dim, hidden_dim):
        super().__init__()
        self.w1 = nn.Linear(in_dim, hidden_dim, bias=False)
        self.v = nn.Linear(in_dim, hidden_dim, bias=False)
        self.w2 = nn.Linear(hidden_dim, in_dim, bias=False)

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.v(x))


class MedLiTSparseMoE(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_experts, k=2):
        super().__init__()
        self.k = k
        self.num_experts = num_experts
        self.gate = nn.Linear(in_dim, num_experts, bias=False)
        self.experts = nn.ModuleList(
            [SwiGLUExpert(in_dim, hidden_dim) for _ in range(num_experts)]
        )

    def forward(self, x):
        B, N, C = x.shape
        x_flat = x.view(-1, C)

        logits = self.gate(x_flat)
        probs = F.softmax(logits, dim=-1)
        topk_weights, topk_indices = torch.topk(probs, self.k, dim=-1)

        importance = probs.sum(0)
        tokens_per_expert = torch.bincount(
            topk_indices.view(-1), minlength=self.num_experts
        ).float()
        l_aux = (
            self.num_experts
            * torch.sum(importance * tokens_per_expert)
            / (B * N * self.k)
        )

        out = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            mask = topk_indices == i
            if mask.any():
                token_idx, k_idx = torch.where(mask)
                expert_out = expert(x_flat[token_idx])
                out[token_idx] += expert_out * topk_weights[token_idx, k_idx].unsqueeze(-1)

        return out.view(B, N, C), l_aux


class GroupedQueryAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=6, num_groups=3):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_groups
        self.head_dim = embed_dim // num_heads
        self.heads_per_group = num_heads // num_groups

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, self.num_groups * self.head_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, self.num_groups * self.head_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

    def forward(self, x):
        B, N, C = x.shape
        q = self.q_proj(x).view(B, N, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, N, self.num_groups, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, N, self.num_groups, self.head_dim).transpose(1, 2)

        k = k.repeat_interleave(self.heads_per_group, dim=1)
        v = v.repeat_interleave(self.heads_per_group, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        attn = F.softmax(scores, dim=-1)
        context = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, N, C)
        return self.out_proj(context)


class MoDRouter(nn.Module):
    def __init__(self, embed_dim, capacity_factor=0.5):
        super().__init__()
        self.capacity_factor = capacity_factor
        self.router_weights = nn.Linear(embed_dim, 1, bias=False)

    def forward(self, x):
        router_logits = self.router_weights(x).squeeze(-1)
        router_probs = torch.sigmoid(router_logits)

        N = x.size(1)
        k = max(1, int(N * self.capacity_factor))
        topk_weights, topk_indices = torch.topk(router_probs, k, dim=-1, sorted=False)
        return topk_weights, topk_indices


class MedLiTBlock(nn.Module):
    def __init__(self, embed_dim, ffn_hidden_dim, num_experts, capacity_factor=0.5):
        super().__init__()
        self.mod_router = MoDRouter(embed_dim, capacity_factor=capacity_factor)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = GroupedQueryAttention(embed_dim)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.moe = MedLiTSparseMoE(embed_dim, ffn_hidden_dim, num_experts)

    def forward(self, x):
        B, N, C = x.shape
        topk_weights, topk_indices = self.mod_router(x)

        batch_idx = (
            torch.arange(B, device=x.device)
            .unsqueeze(-1)
            .expand(-1, topk_indices.size(1))
        )
        selected_x = x[batch_idx, topk_indices]

        attn_out = self.attn(self.ln1(selected_x))
        selected_x = selected_x + attn_out

        moe_out, l_aux = self.moe(self.ln2(selected_x))
        selected_x = selected_x + moe_out

        scaled_out = selected_x * topk_weights.unsqueeze(-1)
        x_out = x.clone()
        x_out[batch_idx, topk_indices] = x[batch_idx, topk_indices] + scaled_out
        return x_out, l_aux


class MedLiT_Seed(nn.Module):
    def __init__(
        self, num_classes=9, img_size=224, patch_size=16, embed_dim=216, capacity_factor=0.5
    ):
        super().__init__()
        self.patch_proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)
        num_patches = (img_size // patch_size) ** 2

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))

        stage_experts = [3, 4, 5]
        ffn_dims = [81, 54, 27]

        self.layers = nn.ModuleList()
        for stage_idx in range(3):
            for _ in range(3):
                self.layers.append(
                    MedLiTBlock(
                        embed_dim=embed_dim,
                        ffn_hidden_dim=ffn_dims[stage_idx],
                        num_experts=stage_experts[stage_idx],
                        capacity_factor=capacity_factor,
                    )
                )

        self.ln = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_proj(x).flatten(2).transpose(1, 2)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) + self.pos_embed

        total_aux_loss = 0.0
        for layer in self.layers:
            x, l_aux = layer(x)
            total_aux_loss += l_aux

        x = self.ln(x[:, 0])
        logits = self.head(x)
        return logits, total_aux_loss


class MedLiTFocalLoss(nn.Module):
    def __init__(self, tau=0.8, gamma=2.0):
        super().__init__()
        self.tau = tau
        self.gamma = gamma

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=-1)
        p_t = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        ce_loss = F.cross_entropy(logits, targets, reduction="none")
        mask = (p_t < self.tau).float()
        focal_weight = (1.0 - p_t) ** self.gamma
        return (mask * focal_weight * ce_loss).mean()


# ---------------------------------------------------------
# 2. Load PathMNIST Test Data Only
# ---------------------------------------------------------
class LocalPathMNISTDataset(Dataset):
    def __init__(self, npz_path, split="test", transform=None):
        npz_file = np.load(npz_path)
        self.images = npz_file[f"{split}_images"]
        self.labels = npz_file[f"{split}_labels"]
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = Image.fromarray(self.images[idx])
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label


info = INFO["pathmnist"]
class_labels = info["label"]
num_classes = len(class_labels)

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

if os.path.exists(NPZ_PATH):
    print(f"Loading PathMNIST test set from: {NPZ_PATH}")
    test_dataset = LocalPathMNISTDataset(NPZ_PATH, split="test", transform=eval_transform)
else:
    print(f"'{NPZ_PATH}' not found locally; loading PathMNIST test set via medmnist...")
    test_dataset = PathMNIST(split="test", transform=eval_transform, download=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)


# ---------------------------------------------------------
# 3. Load Both Pretrained Models (.pth)
# ---------------------------------------------------------
std_model = MedLiT_Seed(
    num_classes=num_classes, img_size=224, embed_dim=216, capacity_factor=0.5
).to(device)
adv_model = MedLiT_Seed(
    num_classes=num_classes, img_size=224, embed_dim=216, capacity_factor=0.5
).to(device)

std_model.load_state_dict(torch.load(STD_CKPT_PATH, map_location=device, weights_only=True))
adv_model.load_state_dict(torch.load(ADV_CKPT_PATH, map_location=device, weights_only=True))

std_model.eval()
adv_model.eval()
print(f"Loaded Standard Model   : {STD_CKPT_PATH}")
print(f"Loaded Adversarial Model: {ADV_CKPT_PATH}")

criterion = MedLiTFocalLoss(tau=0.8, gamma=2.0)


def fgsm_attack(image, epsilon, data_grad):
    perturbed_image = image + epsilon * data_grad.sign()
    return torch.clamp(perturbed_image, -1.0, 1.0)


# ---------------------------------------------------------
# 4. Find "Proving" Samples & Save Comparison Image
#    Criteria: Both right on Clean -> Std fooled by FGSM -> Adv stays right
# ---------------------------------------------------------
attack_criterion = nn.CrossEntropyLoss()

collected_orig = []
collected_noise = []
collected_adv = []
collected_meta = []

for batch_inputs, batch_targets in test_loader:
    batch_inputs = batch_inputs.to(device)
    batch_targets = batch_targets.view(-1).long().to(device)

    # Clean predictions
    with torch.no_grad():
        std_clean_preds = std_model(batch_inputs)[0].argmax(dim=1)
        adv_clean_preds = adv_model(batch_inputs)[0].argmax(dim=1)

    # Generate real FGSM perturbation using CrossEntropyLoss (no tau masking)
    x_req = batch_inputs.clone().detach().requires_grad_(True)
    outputs_std, _ = std_model(x_req)
    loss = attack_criterion(outputs_std, batch_targets)
    std_model.zero_grad()
    loss.backward()

    data_grad = x_req.grad.data
    perturbed_batch = fgsm_attack(batch_inputs, EPSILON, data_grad)

    # Attacked predictions on the exact same adversarial image
    with torch.no_grad():
        std_atk_preds = std_model(perturbed_batch)[0].argmax(dim=1)
        adv_atk_preds = adv_model(perturbed_batch)[0].argmax(dim=1)

    # Filter for samples that prove the Adversarial model's robustness
    proving_mask = (
        (std_clean_preds == batch_targets)
        & (adv_clean_preds == batch_targets)
        & (std_atk_preds != batch_targets)
        & (adv_atk_preds == batch_targets)
    )

    proving_indices = torch.where(proving_mask)[0]
    for idx in proving_indices:
        collected_orig.append(batch_inputs[idx].detach().cpu())
        collected_noise.append(data_grad[idx].sign().detach().cpu())
        collected_adv.append(perturbed_batch[idx].detach().cpu())
        collected_meta.append((
            batch_targets[idx].item(),
            std_clean_preds[idx].item(),
            adv_clean_preds[idx].item(),
            std_atk_preds[idx].item(),
            adv_atk_preds[idx].item(),
        ))
        if len(collected_orig) == NUM_SAMPLES:
            break

    if len(collected_orig) == NUM_SAMPLES:
        break

if len(collected_orig) == 0:
    raise RuntimeError("No proving samples found! Try increasing EPSILON slightly (e.g., 0.08).")

num_found = len(collected_orig)
print(f"Found {num_found} proving sample(s)!")

fig, axes = plt.subplots(num_found, 3, figsize=(14, 4.2 * num_found))
if num_found == 1:
    axes = np.expand_dims(axes, axis=0)

for i in range(num_found):
    orig_np = (collected_orig[i].numpy().transpose(1, 2, 0) + 1.0) / 2.0
    noise_np = (collected_noise[i].numpy().transpose(1, 2, 0) + 1.0) / 2.0
    adv_np = (collected_adv[i].numpy().transpose(1, 2, 0) + 1.0) / 2.0

    y_true, s_c, a_c, s_a, a_a = collected_meta[i]
    true_lbl = class_labels[str(y_true)]
    s_clean = class_labels[str(s_c)]
    a_clean = class_labels[str(a_c)]
    s_atk = class_labels[str(s_a)]
    a_atk = class_labels[str(a_a)]

    # Column 1: Original Image
    axes[i, 0].imshow(np.clip(orig_np, 0, 1))
    axes[i, 0].set_title(
        f"Original Image\n"
        f"True: {true_lbl}\n"
        f"Std Model: {s_clean} [OK]\n"
        f"Adv Model: {a_clean} [OK]",
        fontsize=9.5,
    )
    axes[i, 0].axis("off")

    # Column 2: FGSM Noise
    axes[i, 1].imshow(np.clip(noise_np, 0, 1))
    axes[i, 1].set_title(f"FGSM Noise Pattern\n(epsilon = {EPSILON})", fontsize=10)
    axes[i, 1].axis("off")

    # Column 3: Adversarial Image
    axes[i, 2].imshow(np.clip(adv_np, 0, 1))
    axes[i, 2].set_title(
        f"Adversarial Image (eps={EPSILON})\n"
        f"True: {true_lbl}\n"
        f"Std Model: {s_atk} [WRONG]\n"
        f"Adv Model: {a_atk} [OK]",
        fontsize=9.5,
    )
    axes[i, 2].axis("off")

plt.suptitle(
    "PathMNIST FGSM Demo: Standard vs. Adversarially Trained MedLiT-Seed",
    fontsize=13,
    fontweight="bold",
    y=0.995,
)
plt.tight_layout(rect=[0, 0, 1, 0.98])
plt.savefig(OUTPUT_IMAGE, dpi=300)
plt.close()
print(f"Saved comparison image: {OUTPUT_IMAGE}")

Using device: cpu
Loading PathMNIST test set from: ./v3/pathmnist.npz
Loaded Standard Model   : ./v3/best_medlit_mod_std.pth
Loaded Adversarial Model: ./v3/best_medlit_mod_adv.pth
Found 4 proving sample(s)!
Saved comparison image: pathmnist_fgsm_both_models_demo.png


In [ ]:
# ---------------------------------------------------------
# 5. Full PathMNIST Test Accuracy Summary (Both Models)
# ---------------------------------------------------------
total = 0
std_clean_ok, adv_clean_ok = 0, 0
std_wb_ok, adv_on_std_fgsm_ok, adv_wb_ok = 0, 0, 0

for inputs, targets in tqdm(test_loader, desc=f"Evaluating PathMNIST (eps={EPSILON})"):
    inputs = inputs.to(device)
    targets = targets.view(-1).long().to(device)

    with torch.no_grad():
        std_clean_ok += std_model(inputs)[0].argmax(1).eq(targets).sum().item()
        adv_clean_ok += adv_model(inputs)[0].argmax(1).eq(targets).sum().item()

    # White-box FGSM on Standard Model
    x_std = inputs.clone().detach().requires_grad_(True)
    loss_s = criterion(std_model(x_std)[0], targets)
    std_model.zero_grad()
    loss_s.backward()
    adv_from_std = fgsm_attack(x_std.detach(), EPSILON, x_std.grad.data)

    # White-box FGSM on Adversarial Model
    x_adv = inputs.clone().detach().requires_grad_(True)
    loss_a = criterion(adv_model(x_adv)[0], targets)
    adv_model.zero_grad()
    loss_a.backward()
    adv_from_adv = fgsm_attack(x_adv.detach(), EPSILON, x_adv.grad.data)

    with torch.no_grad():
        std_wb_ok += std_model(adv_from_std)[0].argmax(1).eq(targets).sum().item()
        adv_on_std_fgsm_ok += adv_model(adv_from_std)[0].argmax(1).eq(targets).sum().item()
        adv_wb_ok += adv_model(adv_from_adv)[0].argmax(1).eq(targets).sum().item()

    total += targets.size(0)

print("\n" + "=" * 62)
print(f"       PATHMNIST DEMO RESULTS (FGSM epsilon = {EPSILON})")
print("=" * 62)
print(f"Standard Model    | Clean Acc: {100.0 * std_clean_ok / total:.2f}% | White-Box FGSM Acc: {100.0 * std_wb_ok / total:.2f}%")
print(f"Adversarial Model | Clean Acc: {100.0 * adv_clean_ok / total:.2f}% | Transfer FGSM Acc : {100.0 * adv_on_std_fgsm_ok / total:.2f}% | White-Box FGSM Acc: {100.0 * adv_wb_ok / total:.2f}%")
print("=" * 62)